In [33]:
import cv2
from ultralytics import YOLO
import numpy as np

In [34]:
model = YOLO("yolov8n.pt")

In [35]:
video_path = "traffic2.mp4"
cap = cv2.VideoCapture(video_path)

In [36]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output.mp4", fourcc, 20.0, (640, 480))

In [ ]:
tracker = {}
next_id = 0

total_count = 0
counted_ids = set()

# Direction counts
up_count = 0
down_count = 0

# Lane counts
left_lane_count = 0
right_lane_count = 0

# Line + lane split
line_y = 250
lane_x = 320

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))

    results = model(frame)

    current_objects = []

    #  Frame vehicle count (for density)
    frame_vehicle_count = 0

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])

            if cls in [2, 3, 5, 7]:
                frame_vehicle_count += 1  #  count vehicles in frame 

                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                current_objects.append((cx, cy, x1, y1, x2, y2))

    new_tracker = {}

    for obj in current_objects:
        cx, cy, x1, y1, x2, y2 = obj

        matched_id = None

        for id, prev in tracker.items():
            prev_cx, prev_cy = prev

            distance = abs(cx - prev_cx) + abs(cy - prev_cy)

            if distance < 50:
                matched_id = id
                break

        if matched_id is None:
            matched_id = next_id
            next_id += 1

        new_tracker[matched_id] = (cx, cy)

        # Line crossing logic
        if matched_id in tracker:
            prev_cx, prev_cy = tracker[matched_id]

            crossed_down = (prev_cy < line_y and cy >= line_y)
            crossed_up   = (prev_cy > line_y and cy <= line_y)

            if (crossed_down or crossed_up) and matched_id not in counted_ids:

                total_count += 1
                counted_ids.add(matched_id)

                # Direction
                if crossed_down:
                    down_count += 1
                else:
                    up_count += 1

                # Lane
                if cx < lane_x:
                    left_lane_count += 1
                else:
                    right_lane_count += 1

        # Draw box + ID
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {matched_id}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 2)

    tracker = new_tracker

    #  TRAFFIC DENSITY LOGIC
    if frame_vehicle_count <= 5:
        density = "Low"
        color = (0,255,0)
    elif frame_vehicle_count <= 10:
        density = "Medium"
        color = (0,255,255)
    else:
        density = "High"
        color = (0,0,255)

    # Draw lines
    cv2.line(frame, (0, line_y), (640, line_y), (0,0,255), 2)
    cv2.line(frame, (lane_x, 0), (lane_x, 480), (255,0,0), 2)

    # Display info
    cv2.putText(frame, f"Total: {total_count}", (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.putText(frame, f"Density: {density}", (20,80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    cv2.putText(frame, f"Vehicles in Frame: {frame_vehicle_count}", (20,120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    out.write(frame)


0: 480x640 4 cars, 154.4ms
Speed: 2.8ms preprocess, 154.4ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 cars, 151.9ms
Speed: 2.8ms preprocess, 151.9ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 cars, 128.8ms
Speed: 2.4ms preprocess, 128.8ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 cars, 139.7ms
Speed: 3.5ms preprocess, 139.7ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 cars, 226.7ms
Speed: 3.3ms preprocess, 226.7ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 cars, 132.3ms
Speed: 3.7ms preprocess, 132.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 cars, 153.4ms
Speed: 2.4ms preprocess, 153.4ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 cars, 153.5ms
Speed: 3.1ms preprocess, 153.5ms inference, 2.2ms postprocess per image at shape (1, 3, 48

In [38]:
cap.release()
out.release()

print("Processing Complete")
print(f"Total Vehicles: {total_count}")
print(f"Up Direction: {up_count}")
print(f"Down Direction: {down_count}")
print(f"Left Lane: {left_lane_count}")
print(f"Right Lane: {right_lane_count}")

✅ Processing Complete
🚗 Total Vehicles: 11
⬆️ Up Direction: 0
⬇️ Down Direction: 11
⬅️ Left Lane: 3
➡️ Right Lane: 8
